# Scarlet collaborator reproduction

This notebook is a thin interface to the tested benchmark commands. It reproduces the full declared A/B/C campaign using constrained variable projection and fixed catalog centroids expressed in the recentered-PSF latent frame. The centroid constraint is conditional on independent astrometry; it is not a generic source default. Select or reject numerical solutions only with truth-independent optimality and residual diagnostics, never with injected-source recovery metrics.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

# Configuration: edit only this cell. Run the notebook from the Scarlet checkout.
SCARLET_REPO = Path.cwd().resolve()
if not (SCARLET_REPO / 'setup.py').exists():
    raise RuntimeError('Launch Jupyter from the repository root')
DATA_ROOT = Path('/path/to/collaborator/data')
OUTPUT_ROOT = SCARLET_REPO / 'benchmark_artifacts' / 'collaborator_reproduction'
STARTS = ('A', 'B', 'C')  # Predeclared before inspecting recovery.
MAX_ITER = 1500
FIT_DTYPE = 'float64'
CHANNEL_CHUNK_SIZE = 64
OPTIMALITY_TOLERANCE = 1e-4
OPTIMALITY_CHECK_INTERVAL = 20

In [ ]:
run_dirs = {start: OUTPUT_ROOT / f'start{start}' for start in STARTS}
fit_commands = {}
for start, run_dir in run_dirs.items():
    fit_commands[start] = [
        sys.executable, '-m', 'benchmarks.run_collaborator_reproduction',
        '--data-root', str(DATA_ROOT),
        '--output-dir', str(run_dir),
        '--start', start,
        '--max-iter', str(MAX_ITER),
        '--dtype', FIT_DTYPE,
        '--channel-chunk-size', str(CHANNEL_CHUNK_SIZE),
        '--optimizer', 'variable_projection',
        '--feature', 'centroid_psf',
        '--optimality-tolerance', str(OPTIMALITY_TOLERANCE),
        '--optimality-check-interval', str(OPTIMALITY_CHECK_INTERVAL),
    ]
fit_commands

The next cell performs all three fits and writes NPZ and JSON provenance products. Starts A and B are expected to agree; the known start-C local solution is rejectable from its substantially worse residual without consulting truth.

In [ ]:
for start in STARTS:
    print(f'Running declared start {start}', flush=True)
    subprocess.run(fit_commands[start], cwd=SCARLET_REPO, check=True)

In [ ]:
products = {}
metrics_paths = {}
plot_dirs = {}
for start, run_dir in run_dirs.items():
    products[start] = run_dir / f'scarlet_matched_start{start}.npz'
    metrics_paths[start] = run_dir / 'scarlet_matched_metrics.json'
    plot_dirs[start] = run_dir / 'plots_frame_aligned'
    plot_command = [
        sys.executable, '-m', 'benchmarks.plot_collaborator_reproduction',
        '--product', str(products[start]),
        '--metrics', str(metrics_paths[start]),
        '--output-dir', str(plot_dirs[start]),
    ]
    subprocess.run(plot_command, cwd=SCARLET_REPO, check=True)

In [ ]:
reports = {
    start: json.loads(path.read_text())
    for start, path in metrics_paths.items()
}
observable_summary = {
    start: {
        'iterations': report['iterations'],
        'optimality_converged': report['optimality_converged'],
        'projected_gradient': report['parameter_relative_projected_gradient'],
        'chi_square_per_voxel': report['residual']['chi_square_per_voxel'],
        'lag1_autocorrelation': report['residual']['lag1_autocorrelation'],
    }
    for start, report in reports.items()
}
observable_summary

In [ ]:
recovery_summary = {}
for start, report in reports.items():
    recovery_summary[start] = [
        {
            'spectrum_relative_l2': source['spectrum']['relative_l2'],
            'spectrum_24bin_rms': source['spectrum']['binned_fractional_error_rms'],
            'spectrum_24bin_worst': source['spectrum']['binned_fractional_error_max_abs'],
            'morphology_relative_l2': source['morphology']['relative_l2'],
        }
        for source in report['sources']
    ]
recovery_summary  # Injection-only audit; never use this to select a start.

The full-vector spectral score is flux weighted and can hide weak wavelength intervals. The spectral figures therefore show both the full-resolution converged-minus-truth offset and the 24-bin fractional offset for each source. In the reference run, source 1 contributes only about 2.4% of the blended source flux at 0.6–0.8 µm, with conservative median integrated S/N about 4.6 per channel; its first-bin fractional error is therefore much larger than its overall spectral L2. Source 2 and source 1 above roughly 1.3 µm are recovered much more uniformly.

In [ ]:
from IPython.display import Image, Markdown, display

for start in STARTS:
    display(Markdown(f'## Declared start {start}'))
    for name in (
        'scarlet_collaborator_spectra.png',
        'scarlet_collaborator_morphologies.png',
        'scarlet_collaborator_residual.png',
    ):
        display(Image(filename=str(plot_dirs[start] / name)))